In [25]:
import pandas as pd
import numpy as np
import time
import openmeteo_requests
import sys
import os

src_path = os.path.abspath(os.path.join("..", "src"))
if src_path not in sys.path:
    sys.path.append(src_path)

try:
    from ingestion import fetch_all_cities, fetch_forecast
    from config import CITIES, START_DATE, END_DATE, DAILY_VARIABLES

    print("All modules successfully imported!")

except ModuleNotFoundError as e:
    print(f" Import error: {e}")
    print(" Hint: Make sure there is an __init__.py file inside the src directory.")

All modules successfully imported!


This cell imports all required libraries for the data pipeline.

- pandas, numpy → data processing
- openmeteo_requests → API client for weather data
- sys.path.append("../src") → allows Python to access local modules (config & ingestion scripts)

This ensures a clean project structure with separated logic and configuration.

#### Import Configuration and Ingestion Modules

##### Description

This cell imports all required configuration variables and ingestion functions used in the weather data pipeline.

---

#### Configuration Imports

```python
from config import (
    CITIES,
    START_DATE,
    END_DATE,
    DAILY_VARIABLES,
    ARCHIVE_URL,
    FORECAST_URL
)

In [26]:
historical_data = fetch_all_cities(
    CITIES,
    START_DATE,
    END_DATE,
    DAILY_VARIABLES
)

Fetching → Baku
Baku: 1827 rows
Fetching → Ganja
Ganja: 1827 rows
Fetching → Shusha
Shusha: 1827 rows
Fetching → Nakhchivan
Nakhchivan: 1827 rows


#### Run Full Historical Data Ingestion

#### Description

In this step, we execute the full data ingestion pipeline by calling the `fetch_all_cities` function.

This function retrieves historical weather data for all selected cities using the defined configuration parameters, including the city list and the specified date range.

The result is stored in the `data` variable, which contains a structured dictionary where each key represents a city name and each value is a pandas DataFrame with the corresponding historical weather data.

This step represents the main execution phase of the data extraction process, where raw weather data is collected and prepared for further analysis and processing.

In [27]:
import os

ROOT_DIR = os.path.abspath("..")

RAW_DIR = os.path.join(ROOT_DIR, "data", "raw")

os.makedirs(RAW_DIR, exist_ok=True)

# save
for city, df in historical_data.items():
    file_path = os.path.join(RAW_DIR, f"{city}_historical.csv")
    df.to_csv(file_path, index=False)

print("Saved all files to:", RAW_DIR)

Saved all files to: /Users/samil/Desktop/ITSkillsSprintProjects/weather-intelligence-pipeline-team/data/raw


In [28]:
for city, df in historical_data.items():
    print(f"\n{city}")
    print(f"Rows: {df.shape[0]}")
    print(f"Date range: {df['date'].min()} → {df['date'].max()}")
    


Baku
Rows: 1827
Date range: 2021-04-17 20:00:00 → 2026-04-17 20:00:00

Ganja
Rows: 1827
Date range: 2021-04-17 20:00:00 → 2026-04-17 20:00:00

Shusha
Rows: 1827
Date range: 2021-04-17 20:00:00 → 2026-04-17 20:00:00

Nakhchivan
Rows: 1827
Date range: 2021-04-17 20:00:00 → 2026-04-17 20:00:00


#### Save Raw Historical Data

##### Description

In this step, the ingested historical weather data is saved locally as raw files to ensure data persistence and reproducibility.

First, a directory named `data/raw` is created (if it does not already exist) to store all raw datasets in a structured way.

Then, the pipeline iterates through the `data` dictionary, where each key represents a city name and each value is the corresponding DataFrame containing historical weather records.

For each city, the DataFrame is exported as a CSV file using the city name in lowercase as the filename.

This step ensures that each city's historical dataset is stored separately, making the data easier to manage, access, and use in later stages of the project such as feature engineering and model training.

In [29]:
import os

forecast_data = {}

# project root (notebook harada run olunursa olunsun işləyir)
ROOT_DIR = os.path.abspath("..")

FORECAST_DIR = os.path.join(ROOT_DIR, "data", "raw")
os.makedirs(FORECAST_DIR, exist_ok=True)

for city in CITIES:
    name = city["name"]

    df = fetch_forecast(
        name,
        city["latitude"],
        city["longitude"],
        DAILY_VARIABLES
    )

    forecast_data[name] = df

    file_path = os.path.join(FORECAST_DIR, f"{name}_forecast.csv")
    df.to_csv(file_path, index=False)

print("Forecast saved to:", FORECAST_DIR)

Forecast saved to: /Users/samil/Desktop/ITSkillsSprintProjects/weather-intelligence-pipeline-team/data/raw


### Fetch and Save Forecast Data

#### Description

In this step, the pipeline retrieves **7-day weather forecast data** for all selected cities and stores both in memory and as local files.

A dictionary named `forecast` is created to store the forecast DataFrames, where each key represents a city name and each value contains its corresponding forecast data.

The process iterates through all cities defined in the `CITIES` configuration. For each city, the `fetch_forecast` function is called using its latitude and longitude to retrieve the latest weather predictions.

After fetching the data, the result is:
- stored in the `forecast` dictionary for further processing
- saved locally as a CSV file in the `data/raw` directory

👉 This step ensures that real-time forecast data is available for each city and can be used later for visualization, reporting, or predictive modeling tasks.

### 1. Total Rows Analysis

In [30]:
total_rows = 0

for city, df in historical_data.items():
    print(f"{city}: {df.shape[0]} rows")
    total_rows += df.shape[0]

print("\nTOTAL ROWS IN DATASET:", total_rows)

Baku: 1827 rows
Ganja: 1827 rows
Shusha: 1827 rows
Nakhchivan: 1827 rows

TOTAL ROWS IN DATASET: 7308


### Total Rows Calculation Across All Cities

####  Description

In this step, the total number of ingested records across all cities is calculated and displayed.

The code iterates through the `data` dictionary, where each key represents a city name and each value is a DataFrame containing historical weather data for that city.

For each city, the number of rows (daily records) is printed individually. At the same time, these values are accumulated into a single variable called `total_rows`.

Finally, the total number of rows across all cities is printed.

👉 This step provides a high-level overview of the dataset size and helps validate whether the ingestion process successfully retrieved the expected amount of historical data.

### 2. Date Range Check (requested vs actual)

In [31]:
for city, df in historical_data.items():

    print("\n", city)

    print("Requested start:", START_DATE)
    print("Requested end:", END_DATE)

    print("Actual min date:", df["date"].min())
    print("Actual max date:", df["date"].max())


 Baku
Requested start: 2021-04-18
Requested end: 2026-04-18
Actual min date: 2021-04-17 20:00:00
Actual max date: 2026-04-17 20:00:00

 Ganja
Requested start: 2021-04-18
Requested end: 2026-04-18
Actual min date: 2021-04-17 20:00:00
Actual max date: 2026-04-17 20:00:00

 Shusha
Requested start: 2021-04-18
Requested end: 2026-04-18
Actual min date: 2021-04-17 20:00:00
Actual max date: 2026-04-17 20:00:00

 Nakhchivan
Requested start: 2021-04-18
Requested end: 2026-04-18
Actual min date: 2021-04-17 20:00:00
Actual max date: 2026-04-17 20:00:00


### Date Range Validation per City

#### Description

This step validates whether the ingested data correctly matches the requested time period for each city.

The code iterates through the `data` dictionary, where each key represents a city and each value is a DataFrame containing historical weather data.

For each city, the following information is displayed:
- The requested start and end dates defined in the configuration
- The actual minimum date available in the dataset
- The actual maximum date available in the dataset

👉 This comparison helps verify data completeness and ensures that the API returned the expected historical coverage without missing or truncated time periods.

### 3. Missing Days (Gap Detection)

In [32]:
for city, df in historical_data.items():

    print("\n", city)

    full_range = pd.date_range(df["date"].min(), df["date"].max(), freq="D")

    missing_days = set(full_range) - set(df["date"])

    print("Missing days:", len(missing_days))


 Baku
Missing days: 0

 Ganja
Missing days: 0

 Shusha
Missing days: 0

 Nakhchivan
Missing days: 0


### Missing Days Detection (Data Completeness Check)

#### Description

This step checks the dataset for missing daily records in the time series for each city.

The code generates a complete daily date range between the minimum and maximum dates available in the dataset using `pd.date_range`. This represents the expected full sequence of days without gaps.

Then, it compares this full expected range with the actual dates present in the dataset to identify missing days.

For each city, the total number of missing dates is printed.

👉 This step is important for ensuring time-series integrity, as missing days can negatively affect forecasting models and machine learning performance.

### 4. Null Value Analysis

In [33]:
for city, df in historical_data.items():

    print("\n", city)

    null_counts = df.isnull().sum().sort_values(ascending=False)

    print(null_counts)


 Baku
date                          0
city                          0
weathercode                   0
temperature_2m_max            0
temperature_2m_min            0
apparent_temperature_max      0
apparent_temperature_min      0
precipitation_sum             0
precipitation_hours           0
rain_sum                      0
snowfall_sum                  0
windspeed_10m_max             0
windgusts_10m_max             0
winddirection_10m_dominant    0
dtype: int64

 Ganja
date                          0
city                          0
weathercode                   0
temperature_2m_max            0
temperature_2m_min            0
apparent_temperature_max      0
apparent_temperature_min      0
precipitation_sum             0
precipitation_hours           0
rain_sum                      0
snowfall_sum                  0
windspeed_10m_max             0
windgusts_10m_max             0
winddirection_10m_dominant    0
dtype: int64

 Shusha
date                          0
city                  

### Null Value Analysis

#### Description

This step analyzes missing values (nulls) in the dataset for each city.

The code iterates through the `data` dictionary, where each value is a DataFrame containing historical weather observations.

For each city, it calculates the total number of missing values per column using `isnull().sum()` and sorts the results in descending order to highlight the most affected variables.

👉 This analysis helps identify data quality issues and determines which weather variables may require cleaning, imputation, or further investigation before using the dataset for modeling or analysis.

### 5. Most problematic variables

In [34]:
for city, df in historical_data.items():

    print("\n", city)

    worst_columns = df.isnull().mean().sort_values(ascending=False)

    print("Top missing columns:")
    print(worst_columns.head(5))


 Baku
Top missing columns:
date                  0.0
city                  0.0
weathercode           0.0
temperature_2m_max    0.0
temperature_2m_min    0.0
dtype: float64

 Ganja
Top missing columns:
date                  0.0
city                  0.0
weathercode           0.0
temperature_2m_max    0.0
temperature_2m_min    0.0
dtype: float64

 Shusha
Top missing columns:
date                  0.0
city                  0.0
weathercode           0.0
temperature_2m_max    0.0
temperature_2m_min    0.0
dtype: float64

 Nakhchivan
Top missing columns:
date                  0.0
city                  0.0
weathercode           0.0
temperature_2m_max    0.0
temperature_2m_min    0.0
dtype: float64


### Most Problematic Variables (Highest Missing Rate)

#### Description

This step identifies the weather variables with the highest percentage of missing values in each city's dataset.

The code calculates the proportion of missing values for each column using `isnull().mean()`, which returns the missing rate as a percentage (0 to 1). The results are then sorted in descending order to highlight the most incomplete variables.

For each city, the top 5 variables with the highest missing rates are displayed.

👉 This analysis helps prioritize data cleaning efforts and highlights which features may require imputation, removal, or further validation before modeling.

### 6. Data Quality Summary (FINAL REPORT STYLE)

In [35]:
for city, df in historical_data.items():

    print("\n========================")
    print("CITY:", city)
    print("========================")

    print("Rows:", len(df))
    print("Date range:", df["date"].min(), "→", df["date"].max())

    full_range = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    missing_days = set(full_range) - set(df["date"])

    print("Missing days:", len(missing_days))

    null_ratio = df.isnull().mean().mean()
    print("Overall null ratio:", round(null_ratio * 100, 2), "%")


CITY: Baku
Rows: 1827
Date range: 2021-04-17 20:00:00 → 2026-04-17 20:00:00
Missing days: 0
Overall null ratio: 0.0 %

CITY: Ganja
Rows: 1827
Date range: 2021-04-17 20:00:00 → 2026-04-17 20:00:00
Missing days: 0
Overall null ratio: 0.0 %

CITY: Shusha
Rows: 1827
Date range: 2021-04-17 20:00:00 → 2026-04-17 20:00:00
Missing days: 0
Overall null ratio: 0.0 %

CITY: Nakhchivan
Rows: 1827
Date range: 2021-04-17 20:00:00 → 2026-04-17 20:00:00
Missing days: 0
Overall null ratio: 0.0 %


### Data Quality Summary Report (Per City)

#### Description

This step generates a comprehensive data quality report for each city by combining multiple validation checks into a single overview.

The code iterates through the `data` dictionary, where each value represents a city's historical weather dataset.

For each city, the following metrics are calculated and displayed:

- **Total number of rows** → indicates dataset size  
- **Date range** → shows the actual time coverage of the data  
- **Missing days** → identifies gaps in the daily time series  
- **Overall null ratio** → average percentage of missing values across all variables  

👉 This consolidated view provides a quick but powerful assessment of dataset completeness, consistency, and reliability before moving to feature engineering or model building stages.

#### Task 4 — Data Audit
Still in the notebook, answer these questions with code:

- How many total rows did you ingest?
Total rows ingested

Each city has **1827** rows:

Baku: 1827

Ganja: 1827

Shusha: 1827

Nakhchivan: 1827

📊 Total dataset size

1827 × 4 = **7308** rows


- Are there any gaps in the daily date sequence? (Missing days?)

All cities have 0 missing days.
  
- Are there any null values? Which variables have the most nulls?

All cities don;t have any null values
  
- What is the date range actually covered vs. what you requested?

The dataset is 1 day shifted earlier than requested for all cities:

Requested: 2021-04-18 → 2026-04-18
Actual: 2021-04-17 20:00:00 → 2026-04-17 20:00:00
Conclusion:

There is no data loss, but the range is shifted ~1 day earlier due to timezone/UTC alignment across all cities.
  
- Document every finding — this is your first "trust" checkpoint.